# 任务二：LeNet-5模型对比

本任务将调节实验参数，进行实验对比及分析，包括训练批次大小、迭代轮数、学习速率、最优化方法等。

In [ ]:
import mindspore
# 载入mindspore的默认数据集
import mindspore.dataset as ds
# 常用转化用算子
import mindspore.dataset.transforms.c_transforms as C
# 图像转化用算子
import mindspore.dataset.vision.c_transforms as CV
from mindspore.common import dtype as mstype
# mindspore的tensor
from mindspore import Tensor


# 各类网络层都在nn里面
import mindspore.nn as nn
# 参数初始化的方式
from mindspore.common.initializer import TruncatedNormal
# 设置mindspore运行的环境
from mindspore import context
# 引入训练时候会使用到回调函数，如checkpoint, lossMoniter
from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor, TimeMonitor
# 引入模型
from mindspore.train import Model
# 引入评估模型的包
from mindspore.nn.metrics import Accuracy

# numpy
import numpy as np
# 画图用
import matplotlib.pyplot as plt

# 下载数据相关的包
import os
import requests 
import zipfile

In [ ]:
!wget https://ascend-professional-construction-dataset.obs.cn-north-4.myhuaweicloud.com/ComputerVision/cifar10_mindspore.zip
!unzip cifar10_mindspore.zip

In [ ]:
#创建图像标签列表
category_dict = {0:'airplane',1:'automobile',2:'bird',3:'cat',4:'deer',5:'dog',
                 6:'frog',7:'horse',8:'ship',9:'truck'}
current_path = os.getcwd()
data_path = os.path.join(current_path, 'data/10-verify-bin')
cifar_ds = ds.Cifar10Dataset(data_path)
# 设置图像大小
plt.figure(figsize=(8,8))
i = 1
# 打印9张子图
for dic in cifar_ds.create_dict_iterator():
    plt.subplot(3,3,i)
    plt.imshow(dic['image'].asnumpy())
    plt.xticks([])
    plt.yticks([])
    plt.axis('off')
    plt.title(category_dict[dic['label'].asnumpy().sum()])
    i +=1
    if i > 9 :
        break
plt.show()

In [ ]:
def get_data(datapath):
    cifar_ds = ds.Cifar10Dataset(datapath)
    return cifar_ds

def process_dataset(cifar_ds,batch_size =32,status="train"):
    '''
    ---- 定义算子 ----
    '''
    # 归一化
    rescale = 1.0 / 255.0
    # 平移
    shift = 0.0

    resize_op = CV.Resize((32, 32))
    rescale_op = CV.Rescale(rescale, shift)
    # 对于RGB三通道分别设定mean和std
    normalize_op = CV.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    if status == "train":
        # 随机裁剪
        random_crop_op = CV.RandomCrop([32, 32], [4, 4, 4, 4])
        # 随机翻转
        random_horizontal_op = CV.RandomHorizontalFlip()
    # 通道变化
    channel_swap_op = CV.HWC2CHW()
    # 类型变化
    typecast_op = C.TypeCast(mstype.int32)

    '''
    ---- 算子运算 ----
    '''
    cifar_ds = cifar_ds.map(input_columns="label", operations=typecast_op)
    if status == "train":
        cifar_ds = cifar_ds.map(input_columns="image", operations=random_crop_op)
        cifar_ds = cifar_ds.map(input_columns="image", operations=random_horizontal_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=resize_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=rescale_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=normalize_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=channel_swap_op)
    
    # shuffle
    cifar_ds = cifar_ds.shuffle(buffer_size=1000)
    # 切分数据集到batch_size
    cifar_ds = cifar_ds.batch(batch_size, drop_remainder=True)
    
    return cifar_ds

In [ ]:
"""LeNet."""


def conv(in_channels, out_channels, kernel_size, stride=1, padding=0):
    """weight initial for conv layer"""
    weight = weight_variable()
    return nn.Conv2d(in_channels, out_channels,
                     kernel_size=kernel_size, stride=stride, padding=padding,
                     weight_init=weight, has_bias=False, pad_mode="same")


def fc_with_initialize(input_channels, out_channels):
    """weight initial for fc layer"""
    weight = weight_variable()
    bias = weight_variable()
    return nn.Dense(input_channels, out_channels, weight, bias)


def weight_variable():
    """weight initial"""
    return TruncatedNormal(0.02)


class LeNet5(nn.Cell):
    """
    Lenet network
    

    Args:
        num_class (int): Num classes. Default: 10.

    Returns:
        Tensor, output tensor
    Examples:
        >>> LeNet(num_class=10)

    """
    def __init__(self, num_class=10, channel=3):
        super(LeNet5, self).__init__()
        self.num_class = num_class
        self.conv1 = conv(channel, 6, 5)
        self.conv2 = conv(6, 16, 5)
        self.fc1 = fc_with_initialize(16 * 8 * 8, 120)
        self.fc2 = fc_with_initialize(120, 84)
        self.fc3 = fc_with_initialize(84, self.num_class)
        self.relu = nn.ReLU()      
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        
        

    def construct(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

In [ ]:
from mindspore.train.callback import Callback

class EvalCallBack(Callback):
    def __init__(self, model, eval_dataset, eval_per_epoch, epoch_per_eval):
        self.model = model
        self.eval_dataset = eval_dataset
        self.eval_per_epoch = eval_per_epoch
        self.epoch_per_eval = epoch_per_eval

    def epoch_end(self, run_context):
        cb_param = run_context.original_args()
        cur_epoch = cb_param.cur_epoch_num
        if cur_epoch % self.eval_per_epoch == 0:
            acc = self.model.eval(self.eval_dataset, dataset_sink_mode=False)
            self.epoch_per_eval["epoch"].append(cur_epoch)
            self.epoch_per_eval["acc"].append(acc["Accuracy"])
            print(acc)

## 1. 基准模型训练

首先训练一个基准模型，使用默认参数：batch_size=32, epoch=20, learning_rate=0.001, 优化器=Adam

In [ ]:
# 基准模型参数
batch_size = 32
epochs = 20
learning_rate = 0.001
optimizer_type = "Adam"

# 生成训练数据集
data_path = os.path.join(current_path, 'data/10-batches-bin')
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_baseline", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval)

print("============== Starting Baseline Training ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

## 2. 改变训练批次大小

### 2.1 batch_size = 64

In [ ]:
# 修改batch_size为64
batch_size = 64
epochs = 20
learning_rate = 0.001
optimizer_type = "Adam"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=781, keep_checkpoint_max=10)  # 调整保存步数
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_batch64", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_batch64 = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_batch64)

print("============== Starting Training with batch_size=64 ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

### 2.2 batch_size = 128

In [ ]:
# 修改batch_size为128
batch_size = 128
epochs = 20
learning_rate = 0.001
optimizer_type = "Adam"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=390, keep_checkpoint_max=10)  # 调整保存步数
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_batch128", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_batch128 = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_batch128)

print("============== Starting Training with batch_size=128 ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

## 3. 改变迭代轮数

### 3.1 epoch = 30

In [ ]:
# 修改epoch为30
batch_size = 32
epochs = 30
learning_rate = 0.001
optimizer_type = "Adam"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_epoch30", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_epoch30 = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_epoch30)

print("============== Starting Training with epoch=30 ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

### 3.2 epoch = 40

In [ ]:
# 修改epoch为40
batch_size = 32
epochs = 40
learning_rate = 0.001
optimizer_type = "Adam"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_epoch40", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_epoch40 = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_epoch40)

print("============== Starting Training with epoch=40 ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

## 4. 改变学习速率

### 4.1 learning_rate = 0.005

In [ ]:
# 修改learning_rate为0.005
batch_size = 32
epochs = 20
learning_rate = 0.005
optimizer_type = "Adam"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_lr005", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_lr005 = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_lr005)

print("============== Starting Training with learning_rate=0.005 ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

### 4.2 learning_rate = 0.01

In [ ]:
# 修改learning_rate为0.01
batch_size = 32
epochs = 20
learning_rate = 0.01
optimizer_type = "Adam"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_lr01", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_lr01 = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_lr01)

print("============== Starting Training with learning_rate=0.01 ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

## 5. 改变优化器

### 5.1 使用Momentum优化器

In [ ]:
# 修改优化器为Momentum
batch_size = 32
epochs = 20
learning_rate = 0.01  # Momentum通常使用较高的学习率
optimizer_type = "Momentum"

# 生成训练数据集
cifar_ds = get_data(data_path)
ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

# 构建网络
network = LeNet5(10)

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

# 设置损失函数和优化器
net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
if optimizer_type == "Adam":
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)
elif optimizer_type == "Momentum":
    net_opt = nn.Momentum(params=network.trainable_params(), learning_rate=learning_rate, momentum=0.9)

# 设置回调函数
config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
ckpoint_cb = ModelCheckpoint(prefix="checkpoint_lenet_momentum", directory='./results', config=config_ck)
time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

# 建立模型
model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
eval_per_epoch = 1
epoch_per_eval_momentum = {"epoch": [], "acc": []}
eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval_momentum)

print("============== Starting Training with Momentum optimizer ==============")
model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)

## 6. 模型评估与结果对比

### 6.1 测试集评估

In [ ]:
# 生成测试数据集
data_path_test = os.path.join(current_path, 'data/10-verify-bin')
batch_size_test = 32
cifar_ds_test = ds.Cifar10Dataset(data_path_test)
ds_eval = process_dataset(cifar_ds_test, batch_size=batch_size_test, status="test")

print("测试集评估结果：")
print("1. 基准模型 (batch_size=32, epoch=20, lr=0.001, Adam):")
res_baseline = model.eval(ds_eval, dataset_sink_mode=dataset_sink_mode)
print(res_baseline)

### 6.2 结果可视化

In [ ]:
# 绘制不同batch_size的准确率对比
plt.figure(figsize=(12, 8))

# Batch Size对比
plt.subplot(2, 2, 1)
plt.plot(epoch_per_eval["epoch"], epoch_per_eval["acc"], 'b-', label='batch_size=32')
plt.plot(epoch_per_eval_batch64["epoch"], epoch_per_eval_batch64["acc"], 'g-', label='batch_size=64')
plt.plot(epoch_per_eval_batch128["epoch"], epoch_per_eval_batch128["acc"], 'r-', label='batch_size=128')
plt.title('Accuracy vs. Epoch (Different Batch Sizes)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Epoch对比
plt.subplot(2, 2, 2)
plt.plot(epoch_per_eval["epoch"], epoch_per_eval["acc"], 'b-', label='epoch=20')
plt.plot(epoch_per_eval_epoch30["epoch"], epoch_per_eval_epoch30["acc"], 'g-', label='epoch=30')
plt.plot(epoch_per_eval_epoch40["epoch"], epoch_per_eval_epoch40["acc"], 'r-', label='epoch=40')
plt.title('Accuracy vs. Epoch (Different Epochs)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Learning Rate对比
plt.subplot(2, 2, 3)
plt.plot(epoch_per_eval["epoch"], epoch_per_eval["acc"], 'b-', label='lr=0.001')
plt.plot(epoch_per_eval_lr005["epoch"], epoch_per_eval_lr005["acc"], 'g-', label='lr=0.005')
plt.plot(epoch_per_eval_lr01["epoch"], epoch_per_eval_lr01["acc"], 'r-', label='lr=0.01')
plt.title('Accuracy vs. Epoch (Different Learning Rates)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# Optimizer对比
plt.subplot(2, 2, 4)
plt.plot(epoch_per_eval["epoch"], epoch_per_eval["acc"], 'b-', label='Adam')
plt.plot(epoch_per_eval_momentum["epoch"], epoch_per_eval_momentum["acc"], 'g-', label='Momentum')
plt.title('Accuracy vs. Epoch (Different Optimizers)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

### 6.3 预测结果可视化

In [ ]:
#创建图像标签列表
category_dict = {0:'airplane',1:'automobile',2:'bird',3:'cat',4:'deer',5:'dog',
                 6:'frog',7:'horse',8:'ship',9:'truck'}

cifar_ds = get_data('./data/10-verify-bin')
df_test = process_dataset(cifar_ds,batch_size=1,status='test')

def normalization(data):
    _range = np.max(data) - np.min(data)
    return (data - np.min(data)) / _range

# 设置图像大小
plt.figure(figsize=(10,10))
i = 1
# 打印9张子图
for dic in df_test:
    # 预测单张图片
    input_img = dic[0]    
    output = model.predict(Tensor(input_img))
    output = nn.Softmax()(output)
    # 反馈可能性最大的类别
    predicted = np.argmax(output.asnumpy(),axis=1)[0]
    
    # 可视化
    plt.subplot(3,3,i)
    # 删除batch维度
    input_image = np.squeeze(input_img.asnumpy(),axis=0).transpose(1,2,0)
    # 重新归一化，方便可视化
    input_image = normalization(input_image)
    plt.imshow(input_image)
    plt.xticks([])
    plt.yticks([])
    plt.axis('off')
    plt.title('True label:%s,\n Predicted:%s'%(category_dict[dic[1].asnumpy().sum()],category_dict[predicted]))
    i +=1
    if i > 9 :
        break

plt.show()

## 7. 结论与分析

根据以上实验结果，我们可以分析不同参数对模型性能的影响：

1. **批次大小（Batch Size）**：
   - 较大的批次大小可以提高训练效率，但可能影响模型的泛化能力
   - 过大的批次大小可能导致内存不足

2. **迭代轮数（Epoch）**：
   - 适当的增加迭代轮数可以提高模型精度
   - 过多的迭代轮数可能导致过拟合

3. **学习速率（Learning Rate）**：
   - 较大的学习速率可能导致训练不稳定
   - 较小的学习速率收敛速度慢但可能更稳定

4. **优化器（Optimizer）**：
   - Adam优化器通常收敛更快
   - Momentum优化器在某些情况下可能有更好的泛化性能